# Logistic 回归的贝叶斯后验抽样

> 对应代码：`MathStatsCode/code_in_notes/Chap.MCMC/MCMC_logistic.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：对 Logistic 回归系数的后验分布做随机游走 M-H 抽样，并计算后验均值。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.MCMC`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.MCMC")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

**Logistic回归的Metropolis-Hastings算法示例**

清空环境、关闭分页、固定种子，准备**贝叶斯 Logistic 回归**的后验抽样。


In [ ]:
%%stata
clear all
set more off
set seed 8855


**设定参数**

`local M = 20000`：设定 MCMC 抽样次数（比前几节多，因为参数的维数是3、需要更长的链来充分探索后验）。


In [ ]:
%%stata
local M = 20000


**真值**

设定数据生成过程的真值与样本量：β0=1、β1=1、β2=-1，N=200——模拟用“真值”检验后验均值是否靠近它们。


In [ ]:
%%stata
local beta_0 = 1
local beta_1 = 1
local beta_2 = -1


**样本量**

这一段是完整的 MCMC 后验抽样实现：

**①`Logistic(x)`**：逻辑函数 1/(1+e^{-x})，用 Mata 的逐元素运算符写成 `1:/(1:+exp(-x))`。

**②`gen_logit(N)`**：生成模拟数据——x1~N(1,2²)、x2~χ²(2)（`rchi2(N,1,2)`），线性指数 d*=β0+β1x1+β2x2，d~Bernoulli(Logistic(d*))；返回 N×3 矩阵 (d,x1,x2)。

**③`rho(beta_x,beta_y,Data)`**：计算**接受概率**——对数后验比 = 先验对数密度比（独立标准正态先验：-½Σβ²）+ 对数似然比（对所有观测求和 d·ln F+(1-d)·ln(1-F)，向量化实现）。返回 min(1, e^{对数比})。

**④`q_sampler(beta)`**：随机游走提议——每个系数加 N(0,1²)的扰动。

**⑤`MH_RW(...)`**：MH 算法主体，每轮抽候选点、以 rho 接受/拒绝、记录链状态。注意 rho 作为函数指针传入（与拒绝抽样不同，这里没有独立的工具密度，因为随机游走的对称提议使 q 项抵消）。

**⑥执行**：生成数据、运行2万次 MCMC、丢弃前20%预热样本、计算每个系数的后验均值。结果应接近真值 (1,1,-1)，但带有贝叶斯估计特有的收缩（先验把估计向0拉）。


In [ ]:
%%stata
local N = 200

mata:
// Logistic函数
real matrix Logistic(real matrix x)
{
	return(1:/(1:+exp(-x)))
}

// 产生数据
real matrix gen_logit(real scalar N)
{
	real colvector x1, x2, d_star, p_star, d
	x1 = rnormal(N, 1, 0, 1)*1.414 :+ 1
	x2 = rchi2(N, 1, 2)
	d_star = `beta_0' :+ (`beta_1')*x1 :+ (`beta_2')*x2
	p_star = Logistic(d_star)
	d = runiform(N, 1) :< p_star
	return((d, x1, x2))
}

// 计算接受率
real scalar rho(real rowvector beta_x, real rowvector beta_y, real matrix Data)
{
	real scalar log_post_pai_x, log_post_pai_y, log_ratio, ll_x, ll_y
	real colvector d, F_b_x, F_b_y
	d = Data[., 1]
	// 先验（独立标准正态分布）的对数密度比
	log_post_pai_x = (-beta_x[1]^2-beta_x[2]^2-beta_x[3]^2)/2
	log_post_pai_y = (-beta_y[1]^2-beta_y[2]^2-beta_y[3]^2)/2
	log_ratio = log_post_pai_y - log_post_pai_x
	// 对数似然比
	F_b_x = Logistic(beta_x[1] :+ Data[., 2]:*beta_x[2] :+ Data[., 3]:*beta_x[3])
	F_b_y = Logistic(beta_y[1] :+ Data[., 2]:*beta_y[2] :+ Data[., 3]:*beta_y[3])
	ll_x = sum(d:*log(F_b_x) :+ (1:-d):*log(1:-F_b_x))
	ll_y = sum(d:*log(F_b_y) :+ (1:-d):*log(1:-F_b_y))
	log_ratio = log_ratio + ll_y - ll_x
	return(min((1, exp(log_ratio))))
}

// 随机游走的提议分布
real rowvector q_sampler(real rowvector beta)
{
	return(beta + rnormal(1, 3, 0, 0.1))
}

// 随机游走的MCMC算法，输入：
//    N_samples      : 抽样次数
//   rho(x,y,Data)   : 计算接受率
//   q_sampler(x)    : 给定x，从q中抽样的函数
//      x0           : 初始值
//      Data         : 数据
real matrix MH_RW(real scalar N_samples, pointer rho, pointer q_sampler,
                  real rowvector x0, real matrix Data)
{
	real matrix X
	real rowvector x, y
	real scalar i, u
	X = J(N_samples, length(x0), .)
	x = x0
	for (i=1; i<=N_samples; i++) {
		y = (*q_sampler)(x)
		u = runiform(1, 1)
		if (u <= (*rho)(x, y, Data)) {
			x = y
		}
		X[i, .] = x
	}
	return(X)
}

// 从后验分布中抽样
Data = gen_logit(`N')
beta_post = MH_RW(`M', &rho(), &q_sampler(), (0,0,0), Data)

// 去掉前20%的burn-in，计算后验均值
burn = floor(`M'*0.2)
sub_beta = beta_post[(burn+1)..`M', .]
mean_beta = mean(sub_beta)
printf("Mean beta0 = %f\n", mean_beta[1])
printf("Mean beta1 = %f\n", mean_beta[2])
printf("Mean beta2 = %f\n", mean_beta[3])
end
